In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## 1 žingsnis: Apibrėžkite Pydantic modelius struktūrizuotiems rezultatams

Šie modeliai apibrėžia **schemą**, kurią agentai grąžins. Naudojant `response_format` su Pydantic užtikrina:
- ✅ Tipiškai saugų duomenų išgavimą
- ✅ Automatinį patvirtinimą
- ✅ Nėra laisvo teksto atsakymų analizės klaidų
- ✅ Lengvą sąlyginius maršrutus pagal laukus


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## 2 žingsnis: Sukurkite viešbučių užsakymo įrankį

Šį įrankį kvies **availability_agent**, kad patikrintų, ar yra laisvų kambarių. Naudojame dekoratorių `@ai_function` tam, kad:
- Konvertuoti Python funkciją į AI kviečiamą įrankį
- Automatiškai sugeneruoti JSON schemą LLM
- Tvarkyti parametrų patikrinimą
- Įgalinti automatinį kvietimą agentų

Šiam demonstravimui:
- **Stokholmas, Sietlas, Tokijas, Londonas, Amsterdamas** → Yra kambariai ✅
- **Visi kiti miestai** → Nėra kambarių ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## 3 žingsnis: Apibrėžkite sąlygos funkcijas maršrutavimui

Šios funkcijos tikrina agento atsakymą ir nusprendžia, kuriuo keliu eiti darbo procese.

**Pagrindinis modelis:**
1. Patikrinkite, ar žinutė yra `AgentExecutorResponse`
2. Išanalizuokite struktūruotą išvestį (Pydantic modelis)
3. Grąžinkite `True` arba `False`, kad valdyti maršrutavimą

Darbo procesas vertins šias sąlygas **kraštuose**, kad nuspręstų, kurį vykdytoją kviesti toliau.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## 4 žingsnis: Sukurkite pasirinktą rodymo vykdytoją

Vykdytojai yra darbo eigos komponentai, kurie atlieka transformacijas ar šalutinius efektus. Naudojame dekoratorių `@executor`, kad sukurtume pasirinktą vykdytoją, kuris rodo galutinį rezultatą.

**Pagrindinės sąvokos:**
- `@executor(id="...")` – registruoja funkciją kaip darbo eigos vykdytoją
- `WorkflowContext[Never, str]` – tipų užuominos įėjimui/išėjimui
- `ctx.yield_output(...)` – perduoda galutinį darbo eigos rezultatą


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## 5 žingsnis: Įkelkite Aplinkos Kintamuosius

Sukonfigūruokite LLM klientą. Šis pavyzdys veikia su:
- **GitHub Modeliais** (nemokamas lygis su GitHub tokenu)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 6 žingsnis: Sukurkite AI agentus su struktūrizuotais rezultatais

Sukuriame **tris specializuotus agentus**, kiekvienas įvyniotas į `AgentExecutor`:

1. **availability_agent** – Tikrina viešbučio prieinamumą, naudodamas įrankį
2. **alternative_agent** – Siūlo alternatyvius miestus (kai nėra kambarių)
3. **booking_agent** – Skatina rezervuoti (kai kambariai yra prieinami)

**Pagrindinės savybės:**
- `tools=[hotel_booking]` – Pateikia agentui įrankį
- `response_format=PydanticModel` – Priverčia struktūrizuotą JSON išvestį
- `AgentExecutor(..., id="...")` – Įvynioja agentą darbo eigai naudoti


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## 7 žingsnis: Sukurkite darbinį srautą su sąlyginiais kraštais

Dabar naudojame `WorkflowBuilder`, kad sukonstruotume grafą su sąlyginiu maršrutizavimu:

**Darbinio srauto struktūra:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Pagrindiniai metodai:**
- `.set_start_executor(...)` - Nustato įėjimo tašką
- `.add_edge(from, to, condition=...)` - Prideda sąlyginį kraštą
- `.build()` - Užbaigia darbinį srautą


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## 8 žingsnis: Vykdyti 1 testinį atvejį – Miestas BE prieinamumo (Paryžius)

Išbandykime **nėra prieinamumo** scenarijų prašydami viešbučių Paryžiuje (kur mūsų simuliacijoje nėra kambarių).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## 9 veiksmas: Vykdykite 2 bandymo atvejį - Miestas SU prieinamumu (Stokholmas)

Dabar patikrinkime **prieinamumo** kelią, užklausdami viešbučių Stokholme (kur yra kambarių mūsų simuliacijoje).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Pagrindinės išvados ir tolesni žingsniai

### ✅ Ko išmokote:

1. **WorkflowBuilder Šablonas**
   - Naudokite `.set_start_executor()` norėdami apibrėžti įėjimo tašką
   - Naudokite `.add_edge(from, to, condition=...)` sąlyginiam maršruto pasirinkimui
   - Iškvieskite `.build()` norėdami užbaigti darbo eigą

2. **Sąlyginis maršrutavimas**
   - Sąlygų funkcijos nagrinėja `AgentExecutorResponse`
   - Analizuoja struktūrizuotus atsakymus, kad priimtų sprendimus dėl maršruto
   - Grąžina `True` aktyvuoti kraštą, `False` jį praleisti

3. **Įrankių integracija**
   - Naudokite `@ai_function` Python funkcijoms paversti į AI įrankius
   - Agentai automatiškai kviečia įrankius, kai reikia
   - Įrankiai grąžina JSON, kurį agentai gali analizuoti

4. **Struktūrizuoti atsakymai**
   - Naudokite Pydantic modelius saugiam duomenų išgavimui
   - Nustatykite `response_format=MyModel` agentams kurti
   - Analizuokite atsakymus su `Model.model_validate_json()`

5. **Pasirinktiniai Executoriai**
   - Naudokite `@executor(id="...")` sukurti darbo eigos komponentus
   - Executoriai gali transformuoti duomenis arba atlikti šalutinius veiksmus
   - Naudokite `ctx.yield_output()` generuoti darbo eigos rezultatus

### 🚀 Realios pasaulio taikymo sritys:

- **Kelionių užsakymas**: Tikrinkite prieinamumą, siūlykite alternatyvas, lyginkite pasirinkimus
- **Klientų aptarnavimas**: Maršrutizuokite pagal problemos tipą, nuotaiką, prioritetą
- **E-komercija**: Tikrinkite atsargas, siūlykite alternatyvas, apdorokite užsakymus
- **Turinio moderavimas**: Maršrutizuokite pagal toksiškumo balus, vartotojų pranešimus
- **Patvirtinimo darbo eigos**: Maršrutizuokite pagal sumą, vartotojo vaidmenį, rizikos lygį
- **Daugiapakopė apdorojimo eiga**: Maršrutizuokite pagal duomenų kokybę, išsamumą

### 📚 Tolesni žingsniai:

- Pridėti sudėtingesnes sąlygas (daug kriterijų)
- Įgyvendinti ciklus su darbo eigos būsenų valdymu
- Pridėti sub-darbo eigas pakartotinai naudojamiems komponentams
- Integruoti su tikrais API (viešbučių užsakymas, inventoriaus sistemos)
- Pridėti klaidų valdymą ir atsarginius kelius
- Vizualizuoti darbo eigas naudojant integruotus vizualizacijos įrankius


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
